In [2]:
import os
import random
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import logging
from sklearn.metrics import accuracy_score

from transformers import TrainingArguments
from transformers import Trainer, TrainerCallback
from transformers.trainer_pt_utils import _get_learning_rate
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer, GenerationConfig, AutoConfig

# os.environ["TOKENIZERS_PARALLELISM"] = "true"
#os.environ["CUDA_VISIBLE_DEVICES"]= "0"

In [3]:
MODEL_NAME = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,)  # 토크나이저는 학습되어 있는 것으로
tokenizer.model_max_length=8192

In [5]:
print(tokenizer)

GPT2TokenizerFast(name_or_path='LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', vocab_size=102400, model_max_length=8192, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[BOS]', 'eos_token': '[|endofturn|]', 'unk_token': '[UNK]', 'pad_token': '[PAD]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[BOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[EOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("                               ", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	5: AddedToken("                              ", rstrip=False, lstrip=False, single_word=False, normalized=F

In [6]:
tokenizer.eos_token_id

361

In [7]:
tokenizer.vocab.get("[|endofturn|]")

361

## 데이터 준비

In [8]:
!pwd

/home/work/Korean_Spoken_model


In [8]:
import pandas as pd

# pandora = pd.read_csv("/home/user/10TB/Data/PANDORA/Sampling/pandora_randset1(10).csv", index_col = 0)

spoken = pd.read_csv("/home/work/Korean_Spoken/Korean_Allconcat_preprocessed.csv", index_col = 0)
writing = pd.read_csv("/home/work/Korean_Spoken/KoreanSelectedWriting__Allconcat_preprocessed.csv", index_col = 0)

whole_pt_data = pd.concat([spoken, writing], axis=0, ignore_index=True)

In [9]:
def PreTokenizedPackingData(data, tokenizer, max_length=8192):
    examples = []
    buffer = []
    total_length = 0
    eos_id = tokenizer.eos_token_id
    
    # 모든 데이터를 불러와 max_length에 맞게 packing 진행
    for text in tqdm(data):
        tokens = tokenizer.encode(text, add_special_tokens=False)
        
        if len(tokens) > max_length:  # 단일 문장이 max_length 넘으면 truncation
            tokens = tokens[:max_length]
            tokens[-1] = eos_id  # [EOS] 토큰 넣기 
            examples.append(tokens)
            continue
            
        if total_length + len(tokens) +1 > max_length:  # buffer와 현재 문장이 max_length 넘으면 해당 문장은 다음 buffer로
            buffer[-1] += [0 for i in range(max_length-total_length)]  # buffer를 넘기기 전 max_length까지 padding
            examples.append(buffer)
            buffer = []
            total_length = 0
        
        tokens += [eos_id] # [EOS] 토큰 넣기 
        buffer.append(tokens)
        total_length += len(tokens)
        
    if buffer:
        buffer[-1] += [0 for i in range(max_length-total_length)]  # buffer를 넘기기 전 max_length까지 padding
        examples.append(buffer)
        
    return examples

In [10]:
#pretok_whole_pt = PreTokenizedPackingData(whole_pt_data['Text'].sample(frac=0.01, ignore_index=True).to_list(), tokenizer, 8192)

pretok_whole_pt = PreTokenizedPackingData(whole_pt_data['Text'].to_list(), tokenizer, 8192)

print("Packed Data now created.")

 28% 26328338/93502604 [54:08<2:07:09, 8804.20it/s] 

In [10]:
import pickle

# # save
# with open('/home/work/Korean_Spoken/PreTok_whole_spoken+written_list.pkl', 'wb') as f:
#     pickle.dump(pretok_whole_pt, f)
    
# load
with open('/home/work/Korean_Spoken/PreTok_whole_spoken+written_list.pkl', 'rb') as f4:
    train_set_data = pickle.load(f4)
    
print("Save Complete")

Save Complete


In [11]:
train_set_data[0]

[[1137,
  1818,
  362,
  1229,
  15295,
  1353,
  10560,
  853,
  2453,
  10405,
  2662,
  6734,
  362,
  5198,
  1060,
  76876,
  36392,
  7027,
  932,
  15913,
  650,
  1899,
  838,
  6121,
  4137,
  830,
  13724,
  740,
  362,
  361],
 [2030,
  2391,
  9250,
  2662,
  17297,
  362,
  15295,
  6024,
  25387,
  6016,
  21046,
  1547,
  3826,
  98293,
  24914,
  22232,
  1353,
  10560,
  13834,
  1562,
  87286,
  2373,
  740,
  24153,
  634,
  3174,
  97672,
  954,
  13910,
  6734,
  362,
  361],
 [2781,
  4025,
  740,
  362,
  2453,
  4278,
  67110,
  924,
  1119,
  1023,
  740,
  691,
  1992,
  1051,
  955,
  696,
  5210,
  30885,
  8007,
  6129,
  657,
  869,
  2353,
  31247,
  955,
  634,
  2957,
  59985,
  29509,
  4697,
  17297,
  361],
 [8179,
  740,
  1618,
  88832,
  955,
  696,
  36902,
  1254,
  28548,
  5222,
  20379,
  954,
  13910,
  6734,
  362,
  13435,
  1130,
  1060,
  1353,
  6129,
  5797,
  8367,
  21169,
  2599,
  1813,
  6170,
  1347,
  1766,
  1559,
  50747,
  15

In [12]:
# class PackingPretrainDataset(torch.utils.data.Dataset):
#     def __init__(self, data: list, tokenizer, max_length=8192, eos_id = 2):
#         self.tokenizer = tokenizer
#         self.max_length = max_length
#         self.examples = []

#         buffer = []
#         total_length = 0
        
#         # 모든 데이터를 불러와 max_length에 맞게 packing 진행
#         for text in tqdm(data):
#             tokens = tokenizer.encode(text, add_special_tokens=False)
            
#             if len(tokens) > max_length:  # 단일 문장이 max_length 넘으면 truncation
#                 tokens = tokens[:max_length]
#                 tokens[-1] = eos_id  # [EOS] 토큰 넣기 
#                 self.examples.append(tokens)
#                 continue
                
#             if total_length + len(tokens) > max_length:  # buffer와 현재 문장이 max_length 넘으면 해당 문장은 다음 buffer로
#                 buffer[-1] += [0 for i in range(max_length-total_length)]  # buffer를 넘기기 전 max_length까지 padding
#                 self.examples.append(buffer)
#                 buffer = []
#                 total_length = 0
            
#             tokens += [eos_id] # [EOS] 토큰 넣기 
#             buffer.append(tokens)
#             total_length += len(tokens)
            
#         if buffer:
#             buffer[-1] += [0 for i in range(max_length-total_length)]  # buffer를 넘기기 전 max_length까지 padding
#             self.examples.append(buffer)

#     def __len__(self):
#         return len(self.examples)

#     def __getitem__(self, idx):
#         tokens = self.examples[idx]
#         return {"input_ids": tokens}

In [13]:
class PreTokPackingPretrainDataset(torch.utils.data.Dataset):
    def __init__(self, data: list, tokenizer, max_length=8192):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = data

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        tokens = self.examples[idx]
        return {"input_ids": tokens}

In [14]:
# tokenized packed data
tokenized_whole_dataset = PreTokPackingPretrainDataset(train_set_data, tokenizer, max_length=8192)

#tokenized_whole_dataset = PreTokPackingPretrainDataset(pretok_whole_pt, tokenizer, max_length=8192)

In [15]:
print(tokenized_whole_dataset.__len__())
print(tokenized_whole_dataset.__getitem__(10))
print()
print(tokenizer.decode(tokenized_whole_dataset.__getitem__(10)['input_ids'][0]))
print(tokenizer.decode(tokenized_whole_dataset.__getitem__(10)['input_ids'][-1]))

232183
{'input_ids': [[57911, 715, 50138, 1075, 14859, 2373, 3977, 33072, 722, 905, 3497, 698, 1145, 98680, 6954, 1255, 942, 3100, 657, 1058, 905, 785, 657, 924, 30885, 6913, 1047, 21706, 1075, 1151, 634, 1254, 924, 1119, 1023, 740, 7482, 96266, 846, 838, 3108, 1229, 76876, 1104, 732, 5282, 696, 4573, 1060, 77636, 4148, 361], [28919, 12387, 999, 18106, 4933, 30268, 2597, 3703, 732, 1104, 17297, 8449, 11894, 3070, 1548, 4694, 16559, 45984, 905, 773, 22556, 2220, 634, 1047, 6001, 698, 657, 924, 1119, 1023, 740, 361], [1415, 720, 26178, 22643, 13456, 1546, 10096, 730, 2674, 696, 10306, 50559, 773, 15913, 1229, 76876, 3675, 999, 698, 17297, 13435, 1075, 3308, 6114, 41728, 2836, 720, 2220, 720, 87187, 948, 3885, 773, 657, 5940, 858, 924, 1119, 1023, 9983, 22773, 740, 361], [823, 2703, 634, 3441, 3684, 5940, 634, 1058, 905, 7712, 1353, 72834, 97743, 740, 2048, 13456, 720, 9062, 657, 4286, 15283, 853, 1104, 732, 4521, 1043, 28793, 853, 1559, 17297, 361], [8606, 730, 3675, 634, 6205, 3308, 730

## Data collator

In [120]:
from transformers import DataCollatorForLanguageModeling, DataCollatorWithFlattening, DefaultDataCollator

# data_collator = DataCollatorForLanguageModeling(
#     tokenizer=tokenizer, mlm=False, return_tensors='pt'
# )

#data_collator = DataCollatorWithFlattening(return_tensors='pt')

In [121]:
def get_attention_mask_for_packed_sequence(x, token_id, num_heads= 1, eos: bool = True):
    B, T = x.shape
    # eos 토큰의 위치를 찾음. 찾은 후 위치 값에 +1을 해 attend할 시퀀스의 범위에 EOS 토큰도 포함되도록 함.  (전체 batch를 펼쳐 놓았기 때문에 최대 위치는 B*T)
    eos_idx = (x.view(-1) == token_id).nonzero(as_tuple=True)[0] + eos  

    # EOS 토큰 위치와 batch들의 시작 위치를 구함. (batch를 구분해주기 위함)
    eos_idx_expanded = torch.cat([eos_idx, torch.arange(0,B*T+1,T)]).unique().sort()[0]

    # 현재 eos_idx_expanded에 저장된 위치는 batch를 펼쳐 놓았을 때의 위치이므로,
    # 각 batch 안에서의 위치로 바꿈. ex) 8250 - (8250 // 8192) * 8192 = 8250 - 8192 = 58
    normalized_idx = eos_idx_expanded - (eos_idx_expanded // T) * T
    batch_start_mask = (eos_idx_expanded % T == 0) & (eos_idx_expanded != 0)  # 첫 시작은 0이어야 함.
    normalized_idx = torch.where(batch_start_mask & (normalized_idx == 0), T, normalized_idx)  # 시퀀스 내 마지막 위치의 경우 0으로 표시되므로 T로 바꾸어줌

    # batch 내의 여러 시퀀스들의 길이 계산
    reps = normalized_idx[1:] - normalized_idx[:-1]
    reps = torch.where(reps < 1, normalized_idx[1:], reps)  # 혹시 음수가 나왔을 때를 대비한 코드인데 음수가 나올 일이 없을 듯

    # 시퀀스 경계 위치를 각 시퀀스 길이만큼 반복 후 -> (B, 1, T) 변환 -> (B, T, T)가 되도록 복제
    # ex) [2, 2, 5, 5, 5, 9, 9, 9, 9] 
    repeated_idx = torch.repeat_interleave(normalized_idx[1:], reps).view(B,1,T).expand(-1,T,-1)

    # [[0], [1] , ... , [T-1]]를 -> (B, T, T)가 되도록 복제
    # [0, 1, ..., T-1]이 아님. 따라서 1행은 0이 T개 2행에는 1이 T개 ... 이런 식임. 각 배치는 아래처럼 구성됨.
    # [0, 0, 0, ...,]
    # [1, 1, 1, ...,]
    mask_indices = torch.arange(T).view(1,-1,1).expand(B, -1, T)
    
#     print(repeated_idx)
#     print("repeated_idx_shape:", mask_indices.shape)
#     print("mask_indices_shape:", mask_indices.shape)

    # causal mask를 위한 하삼각행렬 생성 후 batch 크기만큼 복제
    mask = torch.ones(T, T, dtype=torch.bool, device=x.device).tril().expand(B, -1, -1)

    # mask_indices와 비교 후 조건 충족 시 False로 바꿈. 시퀀스 경계를 넘는 토큰 attend 못하게
    mask = mask.masked_fill(mask_indices >= repeated_idx, False)
    mask = mask.unsqueeze(1).expand(-1, num_heads, -1, -1)  # shape 4D로 맞추기. (B, 1, T, T)
    return mask

In [122]:
from transformers import DataCollatorForLanguageModeling, DataCollatorWithFlattening, DefaultDataCollator


class FlashAttentionCollator():
    def __init__(self, *args, separator_id=-100, eos_id=0, return_position_ids=True, return_flash_seq_len = False, **kwargs,):
        self.separator_id = separator_id
        self.eos_id = eos_id
        self.return_position_ids = return_position_ids
        self.return_flash_seq_len = return_flash_seq_len

    def __call__(self, batch_samples):
        batch = {"input_ids": [], "labels": [], "position_ids": [], "attention_mask": [], "cu_seqlens": []}

        
        for eb in batch_samples:
            input_ids = []
            labels = []
            position_ids = []
            cu_seqlens = [0]
            total = 0
            tokens = eb["input_ids"]
        
            
            for seq in tokens:  # iteration of sentence list 
                input_ids += seq
                labels += [self.separator_id] + seq[1:] # 첫 토큰은 마스킹
                
                if self.return_position_ids:
                    position_ids += list(range(len(seq)))
                
                total += len(seq)
                cu_seqlens.append(total)
                
            batch['input_ids'].append(torch.tensor(input_ids))
            batch['labels'].append(torch.tensor(labels))
            batch['position_ids'].append(torch.tensor(position_ids))
            batch['attention_mask'].append((torch.tensor(input_ids) != 0).to(torch.float32))  # 0 == [PAD]
            batch['cu_seqlens'].append(torch.tensor(cu_seqlens, dtype=torch.int32))

        whole_input_ids = torch.stack(batch['input_ids'])
        whole_attention_mask = get_attention_mask_for_packed_sequence(whole_input_ids, self.eos_id)
            
        if self.return_flash_seq_len:
            return {
                "input_ids": whole_input_ids,
                "labels": torch.stack(batch['labels']),
                "position_ids": torch.stack(batch['position_ids']),
                "attention_mask": whole_attention_mask,
                "cu_seq_lens_q": batch['cu_seqlens'],
                "cu_seq_lens_k": batch['cu_seqlens'],
                "max_length_q": total,
                "max_length_k": total,
            }
        
        else:
            return {
                "input_ids": whole_input_ids,
                "labels": torch.stack(batch['labels']),
                "position_ids": torch.stack(batch['position_ids']),
                "attention_mask": whole_attention_mask,
            }

In [123]:
data_collator = FlashAttentionCollator(return_tensors='pt', return_flash_seq_len=False, eos_id=tokenizer.eos_token_id)

In [124]:
from torch.utils.data import DataLoader

eval_dataloader = DataLoader(tokenized_whole_dataset, batch_size=4, collate_fn=data_collator)

for batchss in enumerate(eval_dataloader):
    print(batchss)
    print(batchss[1]['attention_mask'].shape)
    print(batchss[1]['attention_mask'][0][0])
    break

(0, {'input_ids': tensor([[ 1137,  1818,   362,  ...,     0,     0,     0],
        [ 5920,  4064,   696,  ...,     0,     0,     0],
        [ 5763, 14178,  2220,  ...,     0,     0,     0],
        [ 5920,  2491,  2764,  ...,     0,     0,     0]]), 'labels': tensor([[ -100,  1818,   362,  ...,     0,     0,     0],
        [ -100,  4064,   696,  ...,     0,     0,     0],
        [ -100, 14178,  2220,  ...,     0,     0,     0],
        [ -100,  2491,  2764,  ...,     0,     0,     0]]), 'position_ids': tensor([[  0,   1,   2,  ...,  81,  82,  83],
        [  0,   1,   2,  ...,  34,  35,  36],
        [  0,   1,   2,  ..., 116, 117, 118],
        [  0,   1,   2,  ...,  24,  25,  26]]), 'attention_mask': tensor([[[[ True, False, False,  ..., False, False, False],
          [ True,  True, False,  ..., False, False, False],
          [ True,  True,  True,  ..., False, False, False],
          ...,
          [False, False, False,  ...,  True, False, False],
          [False, False, Fals

In [125]:
col_test = data_collator([tokenized_whole_dataset.__getitem__(11)])
print(col_test)

print(tokenizer.decode(
    col_test['input_ids'].tolist()[0]))

print(col_test['attention_mask'][0][-7:])
print(col_test['position_ids'])

{'input_ids': tensor([[ 1415,   720, 48964,  ...,     0,     0,     0]]), 'labels': tensor([[ -100,   720, 48964,  ...,     0,     0,     0]]), 'position_ids': tensor([[ 0,  1,  2,  ..., 55, 56, 57]]), 'attention_mask': tensor([[[[ True, False, False,  ..., False, False, False],
          [ True,  True, False,  ..., False, False, False],
          [ True,  True,  True,  ..., False, False, False],
          ...,
          [False, False, False,  ...,  True, False, False],
          [False, False, False,  ...,  True,  True, False],
          [False, False, False,  ...,  True,  True,  True]]]])}
저도 가보지는 않았지만 요즘 시국에 맞게 많이들 가더라고요 사진으로만 봐도 너무 좋아 보이던데요 여자 친구랑 가셨으면 더욱 신나고 즐거웠을 거 같아요[|endofturn|]여자 친구도 저도 영화를 좋아하는데 코로나 때문에 제대로 극장에도 갈 수 없었거든요 집에서 보는 작은 화면이 아니라 압도할 수 있는 큰 화면으로 보고 싶었어요 작지만 소원을 이룬 것 같아서 너무 기뻤어요[|endofturn|]일상을 흔들 만한 일이라 모두 힘들었지만 그래도 힘든 시간을 잘 견뎌냈어요! 자동차극장이라니 상만으로도 정말 매력적인 일을 실행했어요 작다고 하지만 그 소원들을 계속 이뤄간다면 결국 큰 소원과 다를 게 없을 거예요[|endofturn|]사실 자동차 극장을 왜 가는지 이해가 안 됐어요 근데 사방이 막힌 극장보다 여자 친구

In [128]:
import torch.nn.functional as F

dataloader_example = torch.utils.data.DataLoader(tokenized_whole_dataset,
                                                 batch_size=4,
                                                 collate_fn=data_collator)
torch.set_printoptions(profile="default")

for d in dataloader_example:
   # print(d['input_ids'])`
   # print(d['labels'])
    #print(d['position_ids'])
    print(d['input_ids'].shape)
    print(d['labels'].shape)
    print(d['attention_mask'].shape)
    print(d['position_ids'].shape)
    break

torch.Size([4, 8192])
torch.Size([4, 8192])
torch.Size([4, 1, 8192, 8192])
torch.Size([4, 8192])


In [129]:
import torch._inductor.config as inductor_config
torch._inductor.list_options()

['TYPE_CHECKING',
 'enable_auto_functionalized_v2',
 'debug',
 'disable_progress',
 'verbose_progress',
 'fx_graph_cache',
 'fx_graph_remote_cache',
 'bundle_triton_into_fx_graph_cache',
 'autotune_local_cache',
 'autotune_remote_cache',
 'bundled_autotune_remote_cache',
 'force_disable_caches',
 'sleep_sec_TESTING_ONLY',
 'custom_op_default_layout_constraint',
 'triton_kernel_default_layout_constraint',
 'cpp_wrapper',
 'c_shim_version',
 'dce',
 'static_weight_shapes',
 'size_asserts',
 'nan_asserts',
 'pick_loop_orders',
 'inplace_buffers',
 'allow_buffer_reuse',
 'memory_planning',
 'memory_pool',
 'benchmark_harness',
 'epilogue_fusion',
 'epilogue_fusion_first',
 'pattern_matcher',
 'b2b_gemm_pass',
 'post_grad_custom_pre_pass',
 'post_grad_custom_post_pass',
 'joint_custom_pre_pass',
 'joint_custom_post_pass',
 'pre_grad_custom_pass',
 '_pre_fusion_custom_pass',
 'split_cat_fx_passes',
 'efficient_conv_bn_eval_fx_passes',
 'is_predispatch',
 'group_fusion',
 'batch_fusion',
 'pr

In [131]:
torch._inductor.list_mode_options()

{'default': {},
 'reduce-overhead': {'triton.cudagraphs': True},
 'max-autotune-no-cudagraphs': {'max_autotune': True,
  'coordinate_descent_tuning': True},
 'max-autotune': {'max_autotune': True,
  'triton.cudagraphs': True,
  'coordinate_descent_tuning': True}}

### STTrainer Packing

In [7]:
from transformers import AutoTokenizer
from datasets import Dataset
import trl
import torch
from trl import pack_dataset
from trl.trainer.sft_trainer import DataCollatorForLanguageModeling

tokenizer = AutoTokenizer.from_pretrained("LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct")  # 토크나이저는 학습되어 있는 것으로
tokenizer.model_max_length=8192

text_data = {"inputs": ["여자 친구도 저도 영화를 좋아하는데 코로나 때문에 제대로 극장에도 갈 수 없었거든요 집에서 보는 작은 화면이 아니라 압도할 수 있는 큰 화면으로 보고 싶었어요 작지만 소원을 이룬 것 같아서 너무 기뻤어요",
                        '일상을 흔들 만한 일이라 모두 힘들었지만 그래도 힘든 시간을 잘 견뎌냈어요! 자동차극장이라니 상만으로도 정말 매력적인 일을 실행했어요 작다고 하지만 그 소원들을 계속 이뤄간다면 결국 큰 소원과 다를 게 없을 거예요',
                        '사실 자동차 극장을 왜 가는지 이해가 안 됐어요 근데 사방이 막힌 극장보다 여자 친구랑 둘이 자동차 안에 있으니 둘만의 세상인 것 같았어요 여자 친구가 좋아하는 모습을 보니 저도 기쁘더라고요',
                        '사랑하는 사람의 기쁨이 더 큰 기쁨으로 다가올 때가 있잖아요 저는 전해 듣기만 해도 기분이 좋아집니다 정말 즐거운 주말이었을 것 같아요',
                        '네 광릉수목원 근처라 가는 길도 너무 아름답고 파릇한 나무들을 보며 기분 전환도 됐어요 좋아하는 영화를 자동차라는 공간에서 새롭게 볼 수 있어서 특히 기억에 남을 것 같아요!',
                        '영화도 재밌었겠지만 여자 친구랑 새로운 경험을 하며 즐겁게 즐길 수 있다는 것도 굉장한 행복이죠 주어져도 즐기지 못하면 소용없는데 감정화자씨는 순간을 즐기려고 하는 것 같아요 회사에서 복잡한 업무를 배정받아도 한번 해보겠다고 하는 모습이 늘 대단해 보였어요',
                       ]}

examples = {"input_ids": [],
    "attention_mask": [],
           }

for t in text_data['inputs']:
    examples['input_ids'].append(tokenizer(t)['input_ids'])
    examples['attention_mask'].append(tokenizer(t)['attention_mask'])

print(examples)
    
    
dataset = Dataset.from_dict(examples)
packed_dataset = pack_dataset(dataset, seq_length=256, strategy="bfd")
print(packed_dataset[:])

collator = DataCollatorForLanguageModeling(pad_token_id=0, padding_free=False)

import torch.nn.functional as F

dataloader_example = torch.utils.data.DataLoader(packed_dataset,
                                                 batch_size=4,
                                                 collate_fn=collator)
torch.set_printoptions(profile="default")

for d in dataloader_example:
   # print(d['input_ids'])`
   # print(d['labels'])
    #print(d['position_ids'])
    print(d)
    break

{'input_ids': [[18132, 2430, 720, 1229, 720, 3788, 4605, 7770, 15913, 4358, 1668, 2373, 4701, 21115, 15169, 2361, 868, 1107, 2957, 97743, 740, 1523, 41728, 846, 657, 1662, 732, 9836, 634, 1579, 789, 22838, 2870, 868, 773, 657, 2491, 9836, 13456, 8454, 1559, 2957, 6734, 1662, 2597, 19658, 696, 28781, 924, 1119, 96565, 1562, 43790, 6734], [31862, 696, 9212, 51047, 955, 56415, 2618, 3623, 2957, 2597, 4171, 7543, 1590, 696, 1353, 39221, 5520, 6734, 362, 5577, 70397, 56415, 733, 1142, 1043, 13456, 720, 1990, 7349, 1965, 798, 955, 696, 12944, 2662, 6734, 1662, 3401, 3684, 855, 19658, 1371, 696, 3063, 8702, 1092, 4767, 4950, 2491, 19658, 1548, 16313, 869, 1107, 696, 1022, 3027, 740], [6396, 5577, 21115, 696, 2804, 713, 59985, 4630, 905, 1084, 2389, 6734, 4379, 33479, 634, 35108, 21115, 30885, 4344, 2430, 1339, 3136, 634, 5577, 1084, 2373, 773, 8256, 3136, 1043, 730, 4557, 798, 924, 1119, 19850, 6734, 4344, 2430, 905, 7770, 657, 2674, 696, 846, 733, 1229, 720, 20379, 2560, 3885, 740], [10962, 

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

{'input_ids': [[13905, 720, 6815, 2957, 13910, 2597, 4344, 2430, 1339, 4650, 4503, 696, 691, 1877, 10844, 1060, 10215, 868, 773, 4264, 924, 720, 37536, 1075, 3675, 634, 12099, 901, 12836, 720, 7394, 698, 3627, 838, 63077, 15913, 6954, 1255, 942, 3100, 657, 4922, 696, 7394, 12438, 691, 657, 924, 1119, 1023, 740, 3830, 41728, 12037, 1075, 6296, 4605, 23111, 6912, 1023, 720, 7396, 67157, 13910, 3401, 691, 657, 2674, 634, 3108, 10364, 999, 6106, 6734, 31862, 696, 9212, 51047, 955, 56415, 2618, 3623, 2957, 2597, 4171, 7543, 1590, 696, 1353, 39221, 5520, 6734, 362, 5577, 70397, 56415, 733, 1142, 1043, 13456, 720, 1990, 7349, 1965, 798, 955, 696, 12944, 2662, 6734, 1662, 3401, 3684, 855, 19658, 1371, 696, 3063, 8702, 1092, 4767, 4950, 2491, 19658, 1548, 16313, 869, 1107, 696, 1022, 3027, 740, 18132, 2430, 720, 1229, 720, 3788, 4605, 7770, 15913, 4358, 1668, 2373, 4701, 21115, 15169, 2361, 868, 1107, 2957, 97743, 740, 1523, 41728, 846, 657, 1662, 732, 9836, 634, 1579, 789, 22838, 2870, 868, 77